# WELD-VISION-001 — B3 Free GPU Training

This notebook executes the bounded offline B3 training path only.

Contract:
- YOLOX-Nano only.
- Transfer learning only.
- Train + validation for tuning.
- Frozen 45-image project test must not be used during B3 tuning.
- 1-epoch smoke first; only then one 80-epoch candidate.
- No weights/datasets committed to Git.


In [ ]:
import os, subprocess, json, hashlib, pathlib, sys, platform, time
import torch

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "CUDA GPU is not available in this runtime"

props = torch.cuda.get_device_properties(0)
gpu_info = {
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "vram_bytes": props.total_memory,
}
print(json.dumps(gpu_info, indent=2))


In [ ]:
%cd /content
!rm -rf weld-inspection-ai
!git clone --branch build/mvp-v0.1 --single-branch https://github.com/sjo1848/weld-inspection-ai.git
%cd /content/weld-inspection-ai
!git rev-parse HEAD


In [ ]:
from google.colab import files

uploaded = files.upload()
required = "weld-v0.1-materialized.tar.gz"
assert required in uploaded, f"Upload exactly {required}"
target = pathlib.Path("/content") / required
target.write_bytes(uploaded[required])
print(target, target.stat().st_size, "bytes")


In [ ]:
%cd /content/weld-inspection-ai
!mkdir -p data/materialized
!tar -xzf /content/weld-v0.1-materialized.tar.gz -C data/materialized
!python -m pip install -q -e '.[dev]'
!python ml/scripts/prepare_yolox_dataset.py data/materialized/weld-v0.1 --out data/yolox/weld-v0.1 --link-mode copy
!cat data/yolox/weld-v0.1/yolox-dataset-summary.json

import os
counts = {
    split: sum(1 for p in pathlib.Path(f"/content/weld-inspection-ai/data/yolox/weld-v0.1/{split}2017").iterdir() if p.is_file())
    for split in ("train", "val", "test")
}
print("physical counts:", counts)
assert counts == {"train": 358, "val": 45, "test": 45}


In [ ]:
%cd /content
!rm -rf YOLOX-weld-vendor
!git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git YOLOX-weld-vendor
%cd /content/YOLOX-weld-vendor
!git checkout 6ddff4824372906469a7fae2dc3206c7aa4bbaee
!git rev-parse HEAD

!python -m pip install -q loguru tqdm thop ninja tabulate psutil tensorboard pycocotools opencv-python

import torchvision
print("torchvision:", torchvision.__version__)


In [ ]:
%cd /content/YOLOX-weld-vendor
!mkdir -p weights
!wget -q -O weights/yolox_nano.pth https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

pretrained_sha = sha256("weights/yolox_nano.pth")
print("pretrained sha256:", pretrained_sha)


In [ ]:
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
batch = 8 if vram_gib >= 14 else 4 if vram_gib >= 8 else 2
print("VRAM GiB:", round(vram_gib, 2), "=> batch", batch)

PROJECT = "/content/weld-inspection-ai"
YOLOX = "/content/YOLOX-weld-vendor"
EXP = f"{PROJECT}/ml/yolox/weld_nano_exp.py"
DATA = f"{PROJECT}/data/yolox/weld-v0.1"
CKPT = f"{YOLOX}/weights/yolox_nano.pth"

os.environ["WELD_YOLOX_DATA_DIR"] = DATA
os.environ["WELD_YOLOX_WORKERS"] = "2"


In [ ]:
%cd /content/YOLOX-weld-vendor
os.environ["WELD_YOLOX_EPOCHS"] = "1"

smoke_cmd = (
    "set -o pipefail; "
    f"PYTHONPATH={YOLOX} "
    f"python tools/train.py -f {EXP} -d 1 -b {batch} --fp16 -c {CKPT} "
    "2>&1 | tee /content/b3-smoke.log"
)
print(smoke_cmd)
smoke_started = time.time()
smoke = subprocess.run(["bash", "-lc", smoke_cmd])
smoke_seconds = time.time() - smoke_started
assert smoke.returncode == 0, "B3 smoke failed. Stop here and preserve /content/b3-smoke.log"
print("B3 smoke PASS in", round(smoke_seconds, 1), "seconds")


In [ ]:
%cd /content/YOLOX-weld-vendor
os.environ["WELD_YOLOX_EPOCHS"] = "80"

train_cmd = (
    "set -o pipefail; "
    f"PYTHONPATH={YOLOX} "
    f"python tools/train.py -f {EXP} -d 1 -b {batch} --fp16 -c {CKPT} "
    "2>&1 | tee /content/b3-train.log"
)
print(train_cmd)
train_started = time.time()
train = subprocess.run(["bash", "-lc", train_cmd])
train_seconds = time.time() - train_started
assert train.returncode == 0, "B3 bounded training failed. Preserve /content/b3-train.log"
print("B3 bounded training completed in", round(train_seconds / 60, 2), "minutes")


In [ ]:
%cd /content/YOLOX-weld-vendor
project_sha = subprocess.check_output(
    ["git", "-C", "/content/weld-inspection-ai", "rev-parse", "HEAD"], text=True
).strip()
yolox_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()

best = pathlib.Path("/content/YOLOX-weld-vendor/YOLOX_outputs/weld_nano_v0_1/best_ckpt.pth")
latest = pathlib.Path("/content/YOLOX-weld-vendor/YOLOX_outputs/weld_nano_v0_1/latest_ckpt.pth")

evidence = {
    "status": "B3_SMOKE_PASS_AND_TRAINING_COMPLETE",
    "project_commit": project_sha,
    "yolox_commit": yolox_sha,
    "python": sys.version,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "vram_bytes": torch.cuda.get_device_properties(0).total_memory,
    "batch": batch,
    "epochs": 80,
    "fp16": True,
    "train_images": 358,
    "val_images": 45,
    "frozen_test_images": 45,
    "pretrained_sha256": pretrained_sha,
    "smoke_seconds": smoke_seconds,
    "train_seconds": train_seconds,
    "best_ckpt": str(best) if best.exists() else None,
    "latest_ckpt": str(latest) if latest.exists() else None,
    "best_ckpt_sha256": sha256(best) if best.exists() else None,
    "note": "Frozen project test was not used for B3 tuning."
}
pathlib.Path("/content/b3-environment.json").write_text(json.dumps(evidence, indent=2))
print(json.dumps(evidence, indent=2))

!mkdir -p /content/b3-evidence
!cp /content/b3-smoke.log /content/b3-evidence/
!cp /content/b3-train.log /content/b3-evidence/
!cp /content/b3-environment.json /content/b3-evidence/
!test ! -f /content/YOLOX-weld-vendor/YOLOX_outputs/weld_nano_v0_1/best_ckpt.pth || cp /content/YOLOX-weld-vendor/YOLOX_outputs/weld_nano_v0_1/best_ckpt.pth /content/b3-evidence/
!test ! -f /content/YOLOX-weld-vendor/YOLOX_outputs/weld_nano_v0_1/latest_ckpt.pth || cp /content/YOLOX-weld-vendor/YOLOX_outputs/weld_nano_v0_1/latest_ckpt.pth /content/b3-evidence/
!tar -czf /content/weld-b3-evidence.tar.gz -C /content b3-evidence
!ls -lh /content/weld-b3-evidence.tar.gz


In [ ]:
from google.colab import files
files.download("/content/weld-b3-evidence.tar.gz")
